# Approach 05  LayerDrop → BI → SVD Bridge → KD Pipeline

Four-stage pipeline with independent ablation of each stage.

## Setup  install & versions

In [1]:
# Setup  install & versions
!pip install -q transformers datasets accelerate pandas matplotlib scipy

import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


zsh:1: command not found: pip


/Users/sumitjha/Downloads/nlp-layer-reduction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1
Transformers: 5.18.0
device: mps


## Reproducibility  seeds

In [2]:
# Reproducibility  seeds
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Models

In [3]:
# Models
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_IDS = [
    "openai-community/gpt2",
    "HuggingFaceTB/SmolLM-135M",
]

tokenizers = {}
for mid in MODEL_IDS:
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tokenizers[mid] = tok


## Dataset

In [4]:
# Dataset
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

def join_nonempty(split):
    return "\n\n".join(x for x in split["text"] if x and x.strip())

train_text = join_nonempty(dataset["train"])[:300000]
validation_text = join_nonempty(dataset["validation"])[:100000]
test_text = join_nonempty(dataset["test"])[:100000]
print(dataset)


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


## Helpers  model architecture

In [5]:
# Helpers  model architecture
def get_layers(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    return model.model.layers

def set_layers(model, blocks):
    blocks = torch.nn.ModuleList(blocks)
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        model.transformer.h = blocks
        model.config.n_layer = len(blocks)
    else:
        model.model.layers = blocks
        model.config.num_hidden_layers = len(blocks)
    return model

def load_fresh_model(model_id):
    model = AutoModelForCausalLM.from_pretrained(model_id)
    model.config.use_cache = False
    model.eval()
    return model

def hidden_size(model):
    cfg = model.config
    return getattr(cfg, "n_embd", None) or cfg.hidden_size


## Metrics  PPL / params / latency

In [6]:
# Metrics  PPL / params / latency
import math, time, gc

EVAL_BLOCK_SIZE = 256
EVAL_TOKENS = 3000

def calculate_ppl(model, text, model_id, block_size=EVAL_BLOCK_SIZE, max_tokens=EVAL_TOKENS):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if max_tokens is not None:
        ids = ids[:max_tokens]
    n_blocks = len(ids) // block_size
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(n_blocks):
            x = ids[i*block_size:(i+1)*block_size].unsqueeze(0).to(device)
            out = model(input_ids=x, labels=x, use_cache=False)
            c = x.numel() - 1
            total_nll += out.loss.item() * c
            total_tokens += c
    return math.exp(total_nll / total_tokens)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_latency(model, text, model_id, seq_len=256, repeat=10):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"][:, :seq_len].to(device)
    with torch.no_grad():
        for _ in range(3):
            _ = model(ids, use_cache=False)
        t0 = time.perf_counter()
        for _ in range(repeat):
            _ = model(ids, use_cache=False)
        t1 = time.perf_counter()
    return 1000.0 * (t1 - t0) / repeat

def free_model(m):
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Step 1  Block-Influence profile

In [7]:
# Step 1  Block-Influence profile
import torch.nn.functional as F

def block_influence(model, model_id, text, n_chars=2000):
    model.eval()
    tok = tokenizers[model_id]
    ids = tok(text[:n_chars], return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        out = model(**ids, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states
    return [1.0 - F.cosine_similarity(hs[i].float(), hs[i+1].float(), dim=-1).mean().item()
            for i in range(len(hs) - 1)]

def redundant_order(model_id, text=train_text):
    m = load_fresh_model(model_id).to(device)
    bis = block_influence(m, model_id, text)
    free_model(m)
    return sorted(range(len(bis)), key=lambda i: bis[i]), bis


## Step 1  Hard removal (baseline op)

In [8]:
# Step 1  Hard removal (baseline op)
def hard_remove(model_id, indices):
    model = load_fresh_model(model_id)
    blocks = list(get_layers(model))
    kept = [b for i, b in enumerate(blocks) if i not in set(indices)]
    set_layers(model, kept)
    for i, layer in enumerate(get_layers(model)):
        if hasattr(layer, "layer_idx"):
            layer.layer_idx = i
        if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "layer_idx"):
            layer.self_attn.layer_idx = i
    return model


## Results  sweep & table

In [9]:
# Results  sweep & table

rows = []
for mid in MODEL_IDS:
    m = load_fresh_model(mid).to(device)
    order, bis = redundant_order(mid)
    k = 2
    t = sorted(order[:k])
    # stage: hard remove only (no drop, no bridge, no kd)
    m_h = hard_remove(mid, t).to(device)
    rows.append({"model": mid, "stage": "prune_only", "ppl": calculate_ppl(m_h, validation_text, mid)})
    free_model(m_h)
    m2 = hard_remove(mid, t).to(device)
    rows.append({"model": mid, "stage": "prune+(todo:bridge)+(todo:kd)", "ppl": calculate_ppl(m2, validation_text, mid)})
    free_model(m2)

import pandas as pd
pd.DataFrame(rows)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 26590.58it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18910.53it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 11967.63it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 11517.58it/s]

,model,stage,ppl
0,openai-community/gpt2,prune_only,47.731284
1,openai-community/gpt2,prune+(todo:bridge)+(todo:kd),47.731284
2,HuggingFaceTB/SmolLM-135M,prune_only,25.609107
3,HuggingFaceTB/SmolLM-135M,prune+(todo:bridge)+(todo:kd),25.609107


## Summary
Results are embedded above. See `approaches/reports/` for the matching report file (same numbering).